# Precious Metals Price and Return Visualization

This notebook visualizes futures and spot prices, then daily log returns, for gold, silver, platinum, and palladium.

Input files:

- `data/processed/precious_metals_price_panel_complete.csv`
- `data/processed/precious_metals_daily_log_returns.csv`


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
PRICE_PATH = PROJECT_ROOT / "data" / "processed" / "precious_metals_price_panel_complete.csv"
RETURN_PATH = PROJECT_ROOT / "data" / "processed" / "precious_metals_daily_log_returns.csv"

METALS = ["gold", "silver", "platinum", "palladium"]
METAL_LABELS = {
    "gold": "Gold",
    "silver": "Silver",
    "platinum": "Platinum",
    "palladium": "Palladium",
}
MARKET_LABELS = {"futures": "Futures", "spot": "Spot"}
MARKET_COLORS = {"futures": "#0072B2", "spot": "#D55E00"}
PANEL_LABELS = ["(a)", "(b)", "(c)", "(d)"]

TEXT_COLOR = "#222222"
AXIS_COLOR = "#333333"
GRID_COLOR = "#D9D9D9"

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.edgecolor": AXIS_COLOR,
        "axes.linewidth": 0.8,
        "axes.grid": True,
        "grid.color": GRID_COLOR,
        "grid.alpha": 0.55,
        "grid.linewidth": 0.6,
        "grid.linestyle": "-",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titlelocation": "left",
        "axes.titleweight": "semibold",
        "axes.titlesize": 10.5,
        "axes.labelsize": 9.5,
        "xtick.labelsize": 8.5,
        "ytick.labelsize": 8.5,
        "xtick.color": TEXT_COLOR,
        "ytick.color": TEXT_COLOR,
        "axes.labelcolor": TEXT_COLOR,
        "text.color": TEXT_COLOR,
        "font.family": "DejaVu Sans",
        "font.size": 9.5,
        "legend.fontsize": 8.8,
        "legend.frameon": False,
        "legend.handlelength": 2.4,
        "lines.solid_capstyle": "round",
    }
)


def format_time_axis(ax: plt.Axes, ylabel: str | None = None) -> None:
    ax.set_axisbelow(True)
    ax.margins(x=0.01)
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.tick_params(axis="both", width=0.7, length=3.0, color=AXIS_COLOR)
    if ylabel:
        ax.set_ylabel(ylabel)


def add_panel_title(ax: plt.Axes, label: str, title: str) -> None:
    ax.set_title(f"{label} {title}", pad=6)


In [ ]:
prices = pd.read_csv(PRICE_PATH, parse_dates=["date"]).sort_values("date")
returns = pd.read_csv(RETURN_PATH, parse_dates=["date"]).sort_values("date")

expected_columns = ["date", *(f"{metal}_{market}" for metal in METALS for market in ["futures", "spot"])]
assert set(expected_columns) == set(prices.columns), prices.columns.tolist()
assert set(expected_columns) == set(returns.columns), returns.columns.tolist()

print(f"Price sample: {prices['date'].min().date()} to {prices['date'].max().date()}, {len(prices):,} rows")
print(f"Return sample: {returns['date'].min().date()} to {returns['date'].max().date()}, {len(returns):,} rows")
prices.head()


## Futures and Spot Prices

Each subplot compares the futures and spot price of one precious metal in its original price level.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12.5, 7.2), sharex=True)
axes = axes.ravel()

for ax, metal, panel_label in zip(axes, METALS, PANEL_LABELS):
    for market in ["futures", "spot"]:
        column = f"{metal}_{market}"
        ax.plot(
            prices["date"],
            prices[column],
            label=MARKET_LABELS[market],
            color=MARKET_COLORS[market],
            linewidth=1.05,
            alpha=0.95,
        )
    add_panel_title(ax, panel_label, METAL_LABELS[metal])
    format_time_axis(ax, ylabel="Price")
    ax.legend(loc="upper left", ncols=2)

fig.suptitle("Precious Metal Futures and Spot Prices", fontsize=12.5, fontweight="semibold", y=0.995)
fig.autofmt_xdate(rotation=0, ha="center")
fig.tight_layout(rect=[0, 0, 1, 0.965])
plt.show()


## Daily Log Returns

The return file contains daily log returns for the same futures and spot series. The horizontal line marks zero return.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12.5, 7.2), sharex=True, sharey=True)
axes = axes.ravel()

for ax, metal, panel_label in zip(axes, METALS, PANEL_LABELS):
    for market in ["futures", "spot"]:
        column = f"{metal}_{market}"
        ax.plot(
            returns["date"],
            returns[column],
            label=MARKET_LABELS[market],
            color=MARKET_COLORS[market],
            linewidth=0.65,
            alpha=0.85,
        )
    ax.axhline(0, color=AXIS_COLOR, linewidth=0.75, alpha=0.75)
    add_panel_title(ax, panel_label, METAL_LABELS[metal])
    format_time_axis(ax, ylabel="Daily log return (%)")
    ax.legend(loc="upper left", ncols=2)

fig.suptitle("Precious Metal Futures and Spot Daily Log Returns", fontsize=12.5, fontweight="semibold", y=0.995)
fig.autofmt_xdate(rotation=0, ha="center")
fig.tight_layout(rect=[0, 0, 1, 0.965])
plt.show()
